# Persian Sentiment Analysis

This project focuses on sentiment analysis of Persian text using Natural Language Processing (NLP) and Machine Learning techniques.

## Project Goal

The goal is to build a machine learning system that can classify Persian text based on its sentiment.

In [ ]:
# Check Python version
import sys

print("Python version:", sys.version)

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn hazm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import hazm

print("All libraries imported successfully! ✅")

All libraries imported successfully! ✅


1. Load Dataset

In [ ]:
!pip install -q datasets
from datasets import load_dataset

dataset = load_dataset("ParsiAI/snappfood-sentiment-analysis")  # اسم دقیق رو باید چک کنیم
df = dataset["train"].to_pandas()
df.head()

README.md:   0%|          | 0.00/447 [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B / 8.93MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv: reconstructing file:   0%|          |  0.00B / 1.42MB            

validation.csv: downloading bytes:           |  0.00B            

...

test.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52110 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8337 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9033 [00:00<?, ? examples/s]

,comment,label,label_id
0,غذا خیلی سرد بود در صورتیکه فاصله ما خیلی کم است,SAD,1.0
1,بهتره بتونیم ران یا سینه رو خودمون انتخاب کنیم,HAPPY,0.0
2,غذا بد بود حالم خیییییلی بده. دل دردو دل پیچه....,SAD,1.0
3,با سلام سابق بر این بسته بندی از کیفیت بهتری ب...,SAD,1.0
4,سلام، خیلی ممنون و متشکرم,HAPPY,0.0


In [ ]:
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))

label
HAPPY    26236
SAD      25874
Name: count, dtype: int64
label
HAPPY    0.503473
SAD      0.496527
Name: proportion, dtype: float64


In [ ]:
# چک نال بودن
print(df.isnull().sum())

# طول کامنت‌ها
df["comment_length"] = df["comment"].astype(str).apply(len)
print(df["comment_length"].describe())

# چند نمونه از هر کلاس
print(df[df["label"] == "HAPPY"]["comment"].sample(3, random_state=1).to_list())
print(df[df["label"] == "SAD"]["comment"].sample(3, random_state=1).to_list())

comment     0
label       0
label_id    0
dtype: int64
count    52110.000000
mean        89.893667
std         77.252140
min         11.000000
25%         39.000000
50%         66.000000
75%        114.000000
max       1700.000000
Name: comment_length, dtype: float64
['خیلی زود اوردن همه سفارشام درست بود', 'پنیر اصلا نداشت و هیچ شباهتی به ژامبون تنوری نداشت بیشتر شبیه ساندویچ سرد بود پیک هم که غذارو با تاخیر اورد', 'غذا مثل همیشه خوب بود فقط نوشیدنی بجای باواریا که سفارش داده بودم افس سیب فرستادن؟؟؟']
['غذا رستوران عالی بود فقط برخورد پیک بد بود پول خرد نداشت چند دقیقه علاف کرد وگرنه غذا عالی بود', 'متاسفنه غذا\u200cها چرب و پر نمک هستند', 'محصول عکس دار فرستادن که مجبور به مرجوعی شدم']


In [ ]:
# چک کردن توزیع با درصدهای بالاتر
print(df["comment_length"].quantile([0.90, 0.95, 0.99, 0.999]))

# نمونه‌ی چند کامنت خیلی طولانی
print(df.sort_values("comment_length", ascending=False)["comment"].head(3).to_list())
for txt in df[df["label"] == "SAD"]["comment"].sample(3, random_state=1):
    print(txt)
    print("---")

0.900    182.000
0.950    237.000
0.990    382.000
0.999    653.455
Name: comment_length, dtype: float64
['اسنپ مثل همیشه عالی این همه راه رو سریع اومد داغ اورد ولی واقعا از رستوران ایتالیایی مثل شما بعیده عذاتون طعم خوبی نده. قیمت رو شده بالا ببرین ولی غذا رو خوش طعم درست کنید و باور کنید اگه غذا رو با کیفیت خوب درست کنید باز هم چندین درصد سود میکنید به علاوه این که مشتری بیشتر و بیشتر جذب میکنید اشکلات غذا ۱) خمیر بد طعم و کلفت ۲) پنیرش واقعا طعم خوبی نمیداد طعم کره میداد ۳) سس باربیکرو هم نداشت فقط مرغ\u200cها یه مقدار طعم باربیکیو میدادن ۴) چی بگم دیگه حتی روی خمیر هم زیر مواد سس نداشت من از یک دوست فنلاندی که آشپزه حرفه\u200cای هست این دستور رو گرفتم تو خونه هم درست کردم طعمش معرکه میشه و هزینه زیادی نداره راه کار: (خمیر نباید اینجوری باشه برای پیتزای ایتالیایی و مثل اینکه توش دارچین هم زده بودین شما همون خمیر مایه رو با ارد که قاطی میکنین یک لایه نسبتا نازک درست کنین بزارین ۵ دقیقه طلایی بشه بعد سس مخصوص (فرمول سری) پیتزا دست ساز که میشه پودر فلفل دلمه دودی رب گوجه مرغوب ترجیحا ف

In [ ]:
import re
from hazm import Normalizer

normalizer = Normalizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # حذف تگ‌های HTML (اگه باشه)
    text = re.sub(r"<.*?>", " ", text)
    # حذف لینک‌ها
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # حذف کاراکترهای عجیب (فقط فارسی، انگلیسی، عدد و علائم اصلی نگه داشته می‌شه)
    text = re.sub(r"[^\u0600-\u06FF\s0-9a-zA-Z.,!?؟]", " ", text)
    # نرمال‌سازی حروف عربی به فارسی (مثلاً ي -> ی ، ك -> ک)
    text = normalizer.normalize(text)
    # جمع کردن فاصله‌های اضافه
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_comment"] = df["comment"].apply(clean_text)

# چک کنیم بعد از پاک‌سازی چیزی خراب نشده باشه
df[["comment", "clean_comment"]].sample(5, random_state=1)

,comment,clean_comment
31919,دفعه دوم بود که از زیگ زاگ غذا میگرفتم هردوبار...,دفعه دوم بود که از زیگ زاگ غذا می‌گرفتم هردوبا...
50396,سایز همبرگر قارچ کمی کوچک بود. سیب زمینی‌های م...,سایز همبرگر قارچ کمی کوچک بود. سیب‌زمینی‌های م...
43109,به جای پنیر چرب روزانه پنیر کم چرب رژیمی فرستا...,به جای پنیر چرب روزانه پنیر کم‌چرب رژیمی فرستا...
36064,ماهی تلخ نمیدونم چه جوری یه ماهی رو تلخ درست م...,ماهی تلخ نمیدونم چه جوری یه ماهی رو تلخ درست م...
12806,پسته درجه یک نیست و توش بسته هم داره هسته زردا...,پسته درجه یک نیست و توش بسته هم داره هسته زردا...


In [ ]:
empty_after_clean = (df["clean_comment"].str.len() == 0).sum()
print("تعداد ردیف‌های خالی بعد از پاک‌سازی:", empty_after_clean)

تعداد ردیف‌های خالی بعد از پاک‌سازی: 0


In [ ]:
# لود کردن val و test هم (اگه فقط train رو گرفتی قبلاً)
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

val_df["clean_comment"] = val_df["comment"].apply(clean_text)
test_df["clean_comment"] = test_df["comment"].apply(clean_text)

print("Train:", df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# ذخیره برای مرحله بعد (مدل LSTM)
df.to_csv("train_clean.csv", index=False)
val_df.to_csv("val_clean.csv", index=False)
test_df.to_csv("test_clean.csv", index=False)

Train: (52110, 5)
Validation: (8337, 4)
Test: (9033, 4)


In [ ]:
import torch
from collections import Counter

# توکنایز ساده (شکستن بر اساس فاصله – چون متن رو قبلاً پاک‌سازی کردیم)
def simple_tokenize(text):
    return text.split()

# ساخت vocabulary از روی داده‌ی train
counter = Counter()
for text in df["clean_comment"]:
    counter.update(simple_tokenize(text))

print("تعداد کل کلمات یکتا (قبل از فیلتر):", len(counter))

# فقط کلماتی که حداقل ۲ بار تکرار شدن رو نگه می‌داریم (حذف نویز و اشتباهات تایپی نادر)
min_freq = 2
vocab_words = [word for word, freq in counter.items() if freq >= min_freq]
print("تعداد کلمات بعد از فیلتر (min_freq=2):", len(vocab_words))

# ساخت دیکشنری کلمه -> عدد ، با دو توکن خاص: <PAD> و <UNK>
word2idx = {"<PAD>": 0, "<UNK>": 1}
for word in vocab_words:
    word2idx[word] = len(word2idx)

vocab_size = len(word2idx)
print("سایز نهایی vocabulary:", vocab_size)

تعداد کل کلمات یکتا (قبل از فیلتر): 35196
تعداد کلمات بعد از فیلتر (min_freq=2): 15574
سایز نهایی vocabulary: 15576


In [ ]:
MAX_LEN = 150

def text_to_indices(text, word2idx, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    indices = [word2idx.get(token, word2idx["<UNK>"]) for token in tokens]
    if len(indices) > max_len:
        indices = indices[:max_len]
    else:
        indices = indices + [word2idx["<PAD>"]] * (max_len - len(indices))
    return indices

# تست روی یک نمونه
sample_idx = text_to_indices(df["clean_comment"].iloc[0], word2idx)
print("طول خروجی:", len(sample_idx))
print("نمونه:", sample_idx[:20])

طول خروجی: 150
نمونه: [2, 3, 4, 5, 6, 7, 8, 9, 3, 10, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=MAX_LEN):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = self.labels.iloc[idx]
        indices = text_to_indices(text, self.word2idx, self.max_len)
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.float)

train_dataset = SentimentDataset(df["clean_comment"], df["label_id"], word2idx)
val_dataset = SentimentDataset(val_df["clean_comment"], val_df["label_id"], word2idx)
test_dataset = SentimentDataset(test_df["clean_comment"], test_df["label_id"], word2idx)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# تست سریع یک batch
batch_x, batch_y = next(iter(train_loader))
print("Batch X shape:", batch_x.shape)
print("Batch Y shape:", batch_y.shape)

Batch X shape: torch.Size([64, 150])
Batch Y shape: torch.Size([64])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)  # *2 چون bidirectional

    def forward(self, x):
        embedded = self.embedding(x)                # (batch, seq_len, embed_dim)
        _, (hidden, _) = self.lstm(embedded)         # hidden: (num_layers*2, batch, hidden_dim)
        # ترکیب آخرین لایه‌ی forward و backward
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.dropout(hidden_cat)
        out = self.fc(out)
        return out.squeeze(1)

model = LSTMClassifier(vocab_size=vocab_size).to(device)
print(model)

# تست سریع با یک batch
test_out = model(batch_x.to(device))
print("Output shape:", test_out.shape)


Device: cuda
LSTMClassifier(
  (embedding): Embedding(15576, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)
Output shape: torch.Size([64])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Device: cuda
GPU name: Tesla T4


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()
            preds = (torch.sigmoid(out) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return total_loss / len(loader), acc, f1
    # ساخت مدل تازه از اول (نه ادامه‌ی مدل قبلی)
import torch.optim as optim
model = LSTMClassifier(vocab_size=vocab_size).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

best_val_f1 = 0
patience = 2
patience_counter = 0
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_lstm_model.pt")
        print("  ✅ New best model saved")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  ⏹️ Early stopping at epoch {epoch+1}")
            break

Epoch 1/10 | Train Loss: 0.4088 | Val Loss: 0.3594 | Val Acc: 0.8469 | Val F1: 0.8558
  ✅ New best model saved
Epoch 2/10 | Train Loss: 0.3214 | Val Loss: 0.3316 | Val Acc: 0.8564 | Val F1: 0.8617
  ✅ New best model saved
Epoch 3/10 | Train Loss: 0.2860 | Val Loss: 0.3295 | Val Acc: 0.8630 | Val F1: 0.8673
  ✅ New best model saved
Epoch 4/10 | Train Loss: 0.2517 | Val Loss: 1.3229 | Val Acc: 0.5674 | Val F1: 0.6963
Epoch 5/10 | Train Loss: 0.2284 | Val Loss: 0.3668 | Val Acc: 0.8563 | Val F1: 0.8618
  ⏹️ Early stopping at epoch 5


In [ ]:
# لود کردن بهترین مدل ذخیره‌شده
best_model = LSTMClassifier(vocab_size=vocab_size).to(device)
best_model.load_state_dict(torch.load("best_lstm_model.pt"))

test_loss, test_acc, test_f1 = evaluate(best_model, test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1-score: {test_f1:.4f}")

Test Loss: 0.3479
Test Accuracy: 0.8522
Test F1-score: 0.8573


In [ ]:
!pip install -q transformers

from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

print("Tokenizer and model loaded successfully ✅")

config.json:   0%|          | 0.00/440 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

Tokenizer and model loaded successfully ✅


In [ ]:
class BertSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=150):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts.iloc[idx]
        label = int(self.labels.iloc[idx])
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_len,
            padding="max_length", return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

bert_train_dataset = BertSentimentDataset(df["clean_comment"], df["label_id"], bert_tokenizer)
bert_val_dataset = BertSentimentDataset(val_df["clean_comment"], val_df["label_id"], bert_tokenizer)
bert_test_dataset = BertSentimentDataset(test_df["clean_comment"], test_df["label_id"], bert_tokenizer)

bert_train_loader = DataLoader(bert_train_dataset, batch_size=16, shuffle=True)
bert_val_loader = DataLoader(bert_val_dataset, batch_size=16, shuffle=False)
bert_test_loader = DataLoader(bert_test_dataset, batch_size=16, shuffle=False)

In [ ]:
from torch.optim import AdamW

def evaluate_bert(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return total_loss / len(loader), acc, f1

optimizer = AdamW(bert_model.parameters(), lr=2e-5)

best_val_f1 = 0
patience = 1
patience_counter = 0
EPOCHS = 4

for epoch in range(EPOCHS):
    bert_model.train()
    total_loss = 0
    for batch in bert_train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(bert_train_loader)
    val_loss, val_acc, val_f1 = evaluate_bert(bert_model, bert_val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(bert_model.state_dict(), "best_bert_model.pt")
        print("  ✅ New best model saved")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  ⏹️ Early stopping at epoch {epoch+1}")
            break

Epoch 1/4 | Train Loss: 0.3279 | Val Loss: 0.3006 | Val Acc: 0.8732 | Val F1: 0.8803
  ✅ New best model saved
Epoch 2/4 | Train Loss: 0.2688 | Val Loss: 0.3041 | Val Acc: 0.8694 | Val F1: 0.8784
  ⏹️ Early stopping at epoch 2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("best_bert_model.pt", "/content/drive/MyDrive/best_bert_model.pt")

Mounted at /content/drive


'/content/drive/MyDrive/best_bert_model.pt'

In [ ]:
best_bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
best_bert_model.load_state_dict(torch.load("best_bert_model.pt"))

test_loss, test_acc, test_f1 = evaluate_bert(best_bert_model, bert_test_loader)
print(f"BERT Test Loss: {test_loss:.4f}")
print(f"BERT Test Accuracy: {test_acc:.4f}")
print(f"BERT Test F1-score: {test_f1:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: HooshvareLab/bert-fa-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	thos

BERT Test Loss: 0.3076
BERT Test Accuracy: 0.8718
BERT Test F1-score: 0.8797


In [ ]:
!pip install -q gradio

import gradio as gr

def predict_sentiment(text):
    cleaned = clean_text(text)
    encoding = bert_tokenizer(
        cleaned, truncation=True, max_length=150,
        padding="max_length", return_tensors="pt"
    ).to(device)

    best_bert_model.eval()
    with torch.no_grad():
        outputs = best_bert_model(**encoding)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    happy_prob = probs[0].item()
    sad_prob = probs[1].item()

    label = "😊 مثبت (HAPPY)" if happy_prob > sad_prob else "😞 منفی (SAD)"
    confidence = max(happy_prob, sad_prob)

    return f"{label}\n\nاطمینان: {confidence*100:.1f}%\n\n(مثبت: {happy_prob*100:.1f}% | منفی: {sad_prob*100:.1f}%)"

demo = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(label="نظر خود را وارد کنید", placeholder="مثلاً: غذا خیلی خوشمزه بود و به‌موقع رسید", lines=3),
    outputs=gr.Textbox(label="نتیجه"),
    title="تحلیل احساسات فارسی 🇮🇷",
    description="این مدل با fine-tune کردن ParsBERT روی نظرات کاربران اسنپ‌فود ساخته شده و نظر شما را به عنوان مثبت یا منفی دسته‌بندی می‌کند.",
    examples=[
        ["غذا خیلی خوشمزه بود و به‌موقع رسید، عالی بود"],
        ["غذا سرد بود و کیفیت پایینی داشت، اصلا راضی نبودم"],
        ["بسته‌بندی خوب بود ولی طعم غذا معمولی بود"]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9743a4aee97e146ed4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import numpy as np

def get_predictions_with_text(model, loader, texts):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

preds, labels, probs = get_predictions_with_text(best_bert_model, bert_test_loader, test_df["clean_comment"])

# ساخت دیتافریم برای بررسی راحت‌تر
results_df = test_df.copy().reset_index(drop=True)
results_df["predicted"] = preds
results_df["actual"] = labels
results_df["confidence"] = probs.max(axis=1)

# نمونه‌های اشتباه، مرتب‌شده بر اساس بالاترین اطمینان (یعنی مدل خیلی مطمئن بوده ولی اشتباه کرده - جالب‌ترین‌ها)
wrong_predictions = results_df[results_df["predicted"] != results_df["actual"]].sort_values("confidence", ascending=False)

print(f"تعداد کل نمونه‌های اشتباه: {len(wrong_predictions)} از {len(results_df)} ({len(wrong_predictions)/len(results_df)*100:.1f}%)")
print()

for i, row in wrong_predictions.head(5).iterrows():
    actual_label = "HAPPY" if row["actual"] == 0 else "SAD"
    predicted_label = "HAPPY" if row["predicted"] == 0 else "SAD"
    print(f"متن: {row['comment']}")
    print(f"برچسب واقعی: {actual_label} | پیش‌بینی مدل: {predicted_label} | اطمینان: {row['confidence']*100:.1f}%")
    print("-" * 80)

تعداد کل نمونه‌های اشتباه: 1158 از 9033 (12.8%)

متن: عالی بود و خوشمزه
برچسب واقعی: SAD | پیش‌بینی مدل: HAPPY | اطمینان: 99.5%
--------------------------------------------------------------------------------
متن: حلیم خوشمزه و داغ
برچسب واقعی: SAD | پیش‌بینی مدل: HAPPY | اطمینان: 99.5%
--------------------------------------------------------------------------------
متن: خیلی خوب مثل همیشه عالی بود و به موقع رسید ممنون
برچسب واقعی: SAD | پیش‌بینی مدل: HAPPY | اطمینان: 99.5%
--------------------------------------------------------------------------------
متن: بسیار خوش طمع و عااالی
برچسب واقعی: SAD | پیش‌بینی مدل: HAPPY | اطمینان: 99.4%
--------------------------------------------------------------------------------
متن: ایده و دیزاین بسیار عالیه واى اى کاش کیک و شیرینى‌ها کمى کم شیرین‌تر باشن همه چیز عالى میشه. ممنون از کوک و اسنپ فود
برچسب واقعی: SAD | پیش‌بینی مدل: HAPPY | اطمینان: 99.4%
--------------------------------------------------------------------------------


In [ ]:
# بررسی می‌کنیم آیا نمونه‌های اشتباه با اطمینان بالا (>90%)، اکثراً به یک سمت (مثلاً پیش‌بینی HAPPY ولی برچسب SAD) هستن
high_confidence_wrong = wrong_predictions[wrong_predictions["confidence"] > 0.90]
print(f"تعداد خطاهای با اطمینان بالای ۹۰٪: {len(high_confidence_wrong)}")
print()
print("توزیع این خطاها:")
print(high_confidence_wrong.groupby(["actual", "predicted"]).size())

تعداد خطاهای با اطمینان بالای ۹۰٪: 126

توزیع این خطاها:
actual  predicted
0       1            69
1       0            57
dtype: int64
